# Supervised Tabular Training

This notebook trains a compact flat classifier on the bundled Wine dataset. It follows the same API as Hello World, but uses a few more numeric inputs and exposes root embeddings during the same run. Use it as a tabular comparison point, not as the main nested-data story.

Import the runtime pieces used in the full training loop: Lightning for optimization and Polars for reading the bundled JSONL records.


In [1]:
import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint

import json2vec as j2v

logger.remove()

The Wine buffer is flat, but it gives enough numeric variation to make a clearer supervised example than handmade records. The model will use four chemistry fields to predict the cultivar.


In [2]:
records = pl.read_ndjson("docs/data/wine.jsonl").head(48)

records.head()

alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280_od315_of_diluted_wines,proline,cultivar
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
14.23,1.71,2.43,15.6,127.0,2.8,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,"""class_0"""
12.37,0.94,1.36,10.6,88.0,1.98,0.57,0.28,0.42,1.95,1.05,1.82,520.0,"""class_1"""
12.86,1.35,2.32,18.0,122.0,1.51,1.25,0.21,0.94,4.1,0.76,1.29,630.0,"""class_2"""
13.2,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.4,1050.0,"""class_0"""
12.33,1.1,2.28,16.0,101.0,2.05,1.09,0.63,0.41,3.27,1.25,1.67,680.0,"""class_1"""


The schema is the architecture. Four `Number` requests feed the root encoder, `cultivar` is the categorical target, and `embed=True` asks the root node to return embeddings after training.


In [3]:
model = j2v.Model.from_schema(
    j2v.Number("alcohol"),
    j2v.Number("malic_acid"),
    j2v.Number("color_intensity"),
    j2v.Number("proline"),
    j2v.Category("cultivar", target=True, max_vocab_size=4, topk=[2]),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    embed=True,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

`PolarsDataModule(...)` reads the schema configuration from the model, so batch size, queries, targets, and tensorfield behavior stay tied to one object.

In [4]:
datamodule = j2v.PolarsDataModule(
    model=model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    chunk_batch_size=32,
    sample_rate=1.0,
)

The tutorial trains for one tiny pass. In a real experiment this is where you would scale epochs, validation splits, callbacks, logging, and checkpointing.

In [5]:
trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=1,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    enable_checkpointing=False,
    limit_train_batches=1,
    limit_val_batches=1,
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.


`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=1` reached.


The Rich display confirms that the target and root embedding are both configured before inference.

In [6]:
model

Model [model] batch_size=8 d_model=16 parameters=26,039 arrays=1 fields=5 targets=1 embeds=1
`-- record [root] embed attention=mha n_layers=1 n_heads=4 n_linear=1
    |-- alcohol [number] active query=[*].alcohol
    |    pooling=query weight=1 p_mask=0 p_prune=0 n_heads=4 n_linear=1
    |    jitter=0 n_bands=8 offset=4 objective=mae
    |-- malic_acid [number] active query=[*].malic_acid
    |    pooling=query weight=1 p_mask=0 p_prune=0 n_heads=4 n_linear=1
    |    jitter=0 n_bands=8 offset=4 objective=mae
    |-- color_intensity [number] active query=[*].color_intensity
    |    pooling=query weight=1 p_mask=0 p_prune=0 n_heads=4 n_linear=1
    |    jitter=0 n_bands=8 offset=4 objective=mae
    |-- proline [number] active query=[*].proline
    |    pooling=query weight=1 p_mask=0 p_prune=0 n_heads=4 n_linear=1
    |    jitter=0 n_bands=8 offset=4 objective=mae
    `-- cultivar [category] active target query=[*].cultivar
         pooling=query weight=1 p_mask=0 p_prune=1 n_heads=4 n_linear=1
         max_vocab_size=4 p_unavailable=0.01 topk=[2]

After training, `predict` returns typed outputs for supervised targets and embeddings from nodes configured with `embed=True`, keyed by schema address.

In [7]:
batch = records.to_dicts()[:3]
pprint(model.predict(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   0.12069251388311386,
│   │   │   │   0.23059390485286713,
│   │   │   │   -0.5824893116950989,
│   │   │   │   -0.0746098980307579,
│   │   │   │   0.18406087160110474,
│   │   │   │   0.1363171637058258,
│   │   │   │   -0.12112046033143997,
│   │   │   │   -0.15200947225093842,
│   │   │   │   -0.32712408900260925,
│   │   │   │   -0.04328862950205803,
│   │   │   │   0.033536072820425034,
│   │   │   │   0.1473657339811325,
│   │   │   │   0.44348323345184326,
│   │   │   │   0.26595810055732727,
│   │   │   │   -0.3103999197483063,
│   │   │   │   0.0409512035548687
│   │   │   ],
│   │   │   [
│   │   │   │   0.11594932526350021,
│   │   │   │   0.22562003135681152,
│   │   │   │   -0.5599774718284607,
│   │   │   │   -0.05935094878077507,
│   │   │   │   0.1715320348739624,
│   │   │   │   0.1606142818927765,
│   │   │   │   -0.13093486428260803,
│   │   │   │   -0.17148244380950928,
│   │   │   │   -0.34930306673049927,
│   │   │   │   -0.06923247128725052,
│   │   │   │   0.04945848882198334,
│   │   │   │   0.14247263967990875,
│   │   │   │   0.4561956524848938,
│   │   │   │   0.2630501985549927,
│   │   │   │   -0.2964986562728882,
│   │   │   │   0.04485425353050232
│   │   │   ],
│   │   │   [
│   │   │   │   0.11369623243808746,
│   │   │   │   0.25239208340644836,
│   │   │   │   -0.5765462517738342,
│   │   │   │   -0.0467667318880558,
│   │   │   │   0.1691041737794876,
│   │   │   │   0.13389401137828827,
│   │   │   │   -0.12130993604660034,
│   │   │   │   -0.1759652942419052,
│   │   │   │   -0.33720505237579346,
│   │   │   │   -0.0429610013961792,
│   │   │   │   0.023647893220186234,
│   │   │   │   0.16425642371177673,
│   │   │   │   0.4310329854488373,
│   │   │   │   0.2755330502986908,
│   │   │   │   -0.3013325333595276,
│   │   │   │   0.03120032511651516
│   │   │   ]
│   │   ]
│   },
│   'record/cultivar': {
│   │   'state': {
│   │   │   'valued': [0.43543553352355957, 0.44109898805618286, 0.4421565532684326],
│   │   │   'null': [0.20237010717391968, 0.19776740670204163, 0.1964753121137619],
│   │   │   'padded': [0.14032068848609924, 0.14030000567436218, 0.13977330923080444],
│   │   │   'masked': [0.08544855564832687, 0.08676512539386749, 0.0864020586013794],
│   │   │   'other': [0.1364252269268036, 0.13406836986541748, 0.1351928412914276]
│   │   },
│   │   'content': {
│   │   │   'value': ['class_2', 'class_2', 'class_2'],
│   │   │   'probability': [0.383722722530365, 0.3864351212978363, 0.3810601532459259],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'class_2', 'probability': 0.383722722530365},
│   │   │   │   │   {'label': 'class_1', 'probability': 0.3115282654762268}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'class_2', 'probability': 0.3864351212978363},
│   │   │   │   │   {'label': 'class_1', 'probability': 0.3180103898048401}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'class_2', 'probability': 0.3810601532459259},
│   │   │   │   │   {'label': 'class_1', 'probability': 0.31526607275009155}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}